# Stress DSA — Output 3-1 / 3-2

Edit Input 6 / 7 in Excel, save, then `load_core` + `load_stress`.

`run_standard_*_stress` returns a **dict** of scenario id → stressed book.
Use `output_31_table` / `output_32_table` for the full Excel geometry, or
`stress_*_panel(book)` for one scenario.

See `docs/08-stress-dsa.qmd`.


In [ ]:
from __future__ import annotations

from pathlib import Path

from lic_dsf.load import load_core, load_rating, load_stress
from lic_dsf.output import output_31_table, output_32_table, stress_external_panel
from lic_dsf.stress import run_standard_external_stress, run_standard_public_stress

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent
WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"
WORKBOOK


In [ ]:
# Edit Input 6 / 7 in Excel, save, then reload.
macro, external, ext_base, pub_base = load_core(WORKBOOK)
stress = load_stress(WORKBOOK)
rating = load_rating(WORKBOOK)  # thresholds for Output 3-1 bands

external_stress = run_standard_external_stress(
    macro, external, stress.input6, stress.residual
)
public_stress = run_standard_public_stress(
    macro, external, stress.input6, stress.residual
)
list(external_stress), list(public_stress)


## One scenario panel (B1 GDP)


In [ ]:
stress_external_panel(external_stress["B1_GDP"]).loc[
    :, macro.inputs.first_projection_year : macro.inputs.first_projection_year + 5
]


## Output 3-1 — External stress table


In [ ]:
out_3_1 = output_31_table(
    ext_base,
    external_stress=external_stress,
    thresholds=rating.ci.thresholds.as_dict(),
)
out_3_1.head(20)


## Output 3-2 — Public stress table


In [ ]:
out_3_2 = output_32_table(
    pub_base,
    public_stress=public_stress,
    public_threshold=rating.ci.thresholds.public_pv_debt_to_gdp,
)
out_3_2.head(20)
